# WebSocket 實時數據流
透過 Socket.IO 訂閱設備數據，持續打印原始數據直到手動停止（Ctrl+C / 中斷 kernel）。

## 安裝依賴

In [ ]:
# pip install "python-socketio[client]" requests
# pip uninstall socketio -y
# pip install "python-socketio[client]"

## 1. 設定連線參數

In [ ]:
SERVER   = "https://isensing-s1.u-aizu.ac.jp"
USERNAME = "admin"
PASSWORD = "admin"
DN       = "DCDA0C40188C"

## 2. 取得 Bearer Token

In [5]:
import requests

resp = requests.post(
    f"{SERVER}/api/token",
    json={"username": USERNAME, "password": PASSWORD},
)
resp.raise_for_status()
token = resp.json()["token"]
expires_in = resp.json()["expires_in"]
print(f"✓ token 取得成功，有效期 {expires_in // 3600} 小時")
print(f"  token[:40] = {token[:40]}...")

✓ token 取得成功，有效期 24 小時
  token[:40] = eyJzc29faWQiOiJhZG1pbiJ9.adIXxQ.IODD-vGt...


## 3. WebSocket 實時流（收集數據到列表）

In [ ]:
import socketio
import threading
from datetime import datetime

_stop_event = threading.Event()
sio = socketio.Client(logger=False, engineio_logger=False)

def _ts():
    return datetime.now().strftime("%H:%M:%S.%f")[:-3]

@sio.event
def connect():
    print(f"[{_ts()}] 已連線，訂閱 DN={DN}")
    sio.emit("subscribe", {"dn": DN})

@sio.on("subscribed")
def on_subscribed(data):
    print(f"[{_ts()}] 訂閱確認: {data}")

@sio.on("snapshot")
def on_snapshot(data):
    print(f"[{_ts()}] [snapshot] {data}")

@sio.on("update")
def on_update(data):
    print(f"[{_ts()}] [update] {data}")

@sio.on("stream_ended")
def on_stream_ended(data):
    print(f"[{_ts()}] [stream_ended] {data}")
    _stop_event.set()

@sio.on("error")
def on_error(data):
    print(f"[{_ts()}] [error] {data}")
    _stop_event.set()

@sio.event
def disconnect():
    print(f"[{_ts()}] 已斷線")

## 開始接收（中斷 kernel 或 Ctrl+C 停止）

In [ ]:
_stop_event.clear()
sio.connect(SERVER, auth={"token": token})

try:
    _stop_event.wait()
except KeyboardInterrupt:
    pass
finally:
    if sio.connected:
        sio.disconnect()